# Merge HTML, Text, and PDF AcroForm Extractions

Combines extraction outputs into a single dataframe using post-hoc merge
with field-level conditions.

**Merge logic:**
1. Normalize document IDs across all sources
2. Match on normalized ID
3. For each field:
   - Only one source has value → use that value
   - Both sources have values → use source with higher overall fill rate
   - PDF AcroForm is authoritative for radio button fields when available

In [ ]:
import sys
from pathlib import Path

# Add project root so we can import the merge module
sys.path.insert(0, str(Path.cwd().parent))

from merge.merge_extractions import (
    normalize_doc_id, is_empty, compute_fill_rate, merge_two_sources
)

import pandas as pd
import numpy as np

In [ ]:
# ── Configuration ──
HTML_CSV = '../output/html_top_extraction.csv'
TEXT_CSV = '../output/text_top_extraction.csv'
PDF_CSV  = '../output/pdf_acroform_extraction.csv'  # optional
OUTPUT_CSV = '../output/merged_top.csv'

## 1. Load extraction outputs

In [ ]:
df_html = pd.read_csv(HTML_CSV)
df_text = pd.read_csv(TEXT_CSV)
print(f'HTML shape: {df_html.shape}')
print(f'Text shape: {df_text.shape}')

## 2. Merge HTML + Text

In [ ]:
merged = merge_two_sources(df_html, df_text, 'html', 'text')
display(merged.head(10))

## 3. (Optional) Merge with PDF AcroForm

For radio button fields where PDF AcroForm is more reliable, mark them as authoritative:

In [ ]:
import os

if os.path.exists(PDF_CSV):
    df_pdf = pd.read_csv(PDF_CSV)
    print(f'PDF shape: {df_pdf.shape}')

    # Fields where PDF is authoritative (radio buttons)
    pdf_authoritative = [
        'approval_period',
        'waive_1902a',
        'waive_statewideness',
        'statecontracts_mcos1', 'statecontracts_mcos2',
        'statecontracts_mcos3', 'statecontracts_mcos4',
        'payforresidential',
        'reimburse_paidcg',
    ]

    merged = merge_two_sources(
        merged, df_pdf, 'html+text', 'pdf_acroform',
        authoritative_fields=pdf_authoritative,
        authoritative_source='b',
    )
else:
    print('No PDF CSV available, skipping PDF merge.')

## 4. Save merged output

In [ ]:
merged.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(merged)} records to {OUTPUT_CSV}')

## 5. Fill-rate comparison (before vs after merge)

In [ ]:
cols = [c for c in merged.columns if c != 'document_id']
rows = []
for col in cols:
    rows.append({
        'Column': col,
        'HTML Fill%':   f"{100*compute_fill_rate(df_html, col):.1f}%",
        'Text Fill%':   f"{100*compute_fill_rate(df_text, col):.1f}%",
        'Merged Fill%': f"{100*compute_fill_rate(merged, col):.1f}%",
    })
comparison = pd.DataFrame(rows)
display(comparison)